# Phase 3 — Integration E2E Test (TASK-14)

Notebook này chạy 15 câu hỏi thử nghiệm qua toàn bộ pipeline RAG:

```
question → QueryPlanner → SubgraphExtractor → HybridSearch → ContextAssembler → AnswerGenerator
```

**Phạm vi [A]:** Đất đai (chuyển mục đích SDĐ + cấp sổ đỏ lần đầu), TP.HCM + Đồng Nai + Toàn quốc  
**DoD cần đạt:**
- DoD 1: Câu hỏi Đất đai TP.HCM trả lời trong < 30s
- DoD 2: ≥ 2 câu hỏi cho mỗi thủ tục
- DoD 3: Câu hỏi thiếu jurisdiction → `confirmation_needed=True`
- DoD 4: Negative test khai sinh TP.HCM vs Đồng Nai → không bịa sự khác biệt
- DoD 5: Ghi kết quả + nhận xét vào notebook

> **Lưu ý citation**: LLM có thể dùng format tắt `[Điều X, Luật Y]` thay vì format chuẩn `[Điều X, Văn bản Y]`.  
> `parse_citations()` chỉ bắt format chuẩn — citation count = 0 không có nghĩa LLM không trích dẫn.  
> Đánh giá chất lượng trích dẫn bằng mắt qua phần TRẢ LỜI.

In [ ]:
import os
import sys
import json
import time
import logging
from pathlib import Path

# Tìm project root bất kể notebook được chạy từ đâu
_cwd = Path.cwd()
_project_root = next(
    (p for p in [_cwd] + list(_cwd.parents) if (p / 'CLAUDE.md').exists()),
    _cwd,
)
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
print(f'Project root: {_project_root}')

from dotenv import load_dotenv
load_dotenv(_project_root / '.env')

logging.basicConfig(level=logging.WARNING)
print('Import OK')

In [ ]:
import anthropic
from neo4j import GraphDatabase
from qdrant_client import QdrantClient

from src.ingestion.vectorizer import load_model
from src.pipeline import run_pipeline

# Khởi tạo clients một lần, dùng lại cho tất cả câu hỏi
neo4j_driver = GraphDatabase.driver(
    os.getenv('NEO4J_URI', 'bolt://localhost:7687'),
    auth=(os.getenv('NEO4J_USER', 'neo4j'), os.getenv('NEO4J_PASSWORD', '')),
)
qdrant_client = QdrantClient(
    host=os.getenv('QDRANT_HOST', 'localhost'),
    port=int(os.getenv('QDRANT_PORT', '6333')),
)
anthropic_client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
model = load_model()

print('Clients OK')

In [ ]:
results = []  # lưu kết quả để đánh giá cuối

def ask(question: str, label: str = '') -> dict:
    """Chạy pipeline và in kết quả gọn."""
    print(f"\n{'='*70}")
    if label:
        print(f"[{label}]")
    print(f"CÂU HỎI: {question}")
    print('='*70)

    result = run_pipeline(
        question,
        neo4j_driver=neo4j_driver,
        qdrant_client=qdrant_client,
        anthropic_client=anthropic_client,
        model=model,
    )

    if result['confirmation_needed']:
        print('⚠️  CẦN XÁC NHẬN:')
        print(result['confirmation_prompt'])
    else:
        print(f"📊 LCCIDs: {result['lccids_count']}  |  Top-k: {result['top_k_count']}  |  Context: ~{result['context_tokens']} tokens")
        print(f"\n💬 TRẢ LỜI:\n{result['answer']}")
        if result['citations']:
            print(f"\n📌 CITATIONS parsed ({len(result['citations'])}): {result['citations']}")
        else:
            print('\n📌 CITATIONS parsed: 0 (LLM có thể dùng format tắt — kiểm tra thủ công)')

    print(f"\n⏱️  {result['elapsed_seconds']}s")
    return result

print('ask() ready')

## 1. Chuyển mục đích sử dụng đất — TP.HCM (DoD 1 + DoD 2)

In [ ]:
# DoD 1: câu hỏi chuẩn, phải trả lời trong < 30s
r = ask(
    'Điều kiện để chuyển mục đích sử dụng đất tại TP.HCM là gì?',
    label='Q01 | CMĐSDĐ TP.HCM | DoD-1'
)
results.append(r)

# DoD 1 checks
assert r['elapsed_seconds'] < 30, f'TIMEOUT: {r["elapsed_seconds"]}s'
assert not r['confirmation_needed'], 'Câu hỏi có đủ jurisdiction — không được confirmation_needed'
# Citation count là soft check vì LLM có thể dùng format tắt
if len(r['citations']) >= 1:
    print('✅ DoD 1 PASS (có citation theo format chuẩn)')
else:
    print('⚠️  DoD 1 PARTIAL — dưới 30s, có trả lời, nhưng citation format chưa chuẩn (kiểm tra thủ công)')

In [ ]:
r = ask(
    'Hộ gia đình có được chuyển đất nông nghiệp sang đất ở tại TP.HCM không? Cần điều kiện gì?',
    label='Q02 | CMĐSDĐ TP.HCM — điều kiện hộ gia đình'
)
results.append(r)

In [ ]:
r = ask(
    'Hồ sơ xin chuyển mục đích sử dụng đất tại TP.HCM gồm những giấy tờ gì?',
    label='Q03 | CMĐSDĐ TP.HCM — hồ sơ'
)
results.append(r)

In [ ]:
r = ask(
    'Nghĩa vụ tài chính khi chuyển mục đích sử dụng đất sang đất ở tại TP.HCM là bao nhiêu?',
    label='Q04 | CMĐSDĐ TP.HCM — nghĩa vụ tài chính'
)
results.append(r)

## 2. Chuyển mục đích sử dụng đất — Đồng Nai

In [ ]:
r = ask(
    'Quy trình chuyển mục đích sử dụng đất nông nghiệp sang đất ở tại Đồng Nai như thế nào?',
    label='Q05 | CMĐSDĐ Đồng Nai — quy trình'
)
results.append(r)

In [ ]:
r = ask(
    'Thời hạn giải quyết hồ sơ chuyển mục đích sử dụng đất tại Đồng Nai là bao lâu?',
    label='Q06 | CMĐSDĐ Đồng Nai — thời hạn'
)
results.append(r)

## 3. Cấp sổ đỏ lần đầu — TP.HCM (DoD 2)

In [ ]:
r = ask(
    'Điều kiện để được cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM là gì?',
    label='Q07 | Cấp sổ đỏ TP.HCM — điều kiện'
)
results.append(r)

In [ ]:
r = ask(
    'Hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại TP.HCM gồm những gì?',
    label='Q08 | Cấp sổ đỏ TP.HCM — hồ sơ'
)
results.append(r)

In [ ]:
r = ask(
    'Trường hợp đất không có giấy tờ tại TP.HCM thì có được cấp sổ đỏ lần đầu không? Điều kiện là gì?',
    label='Q09 | Cấp sổ đỏ TP.HCM — đất không giấy tờ'
)
results.append(r)

## 4. Cấp sổ đỏ lần đầu — Đồng Nai

In [ ]:
r = ask(
    'Cơ quan nào tiếp nhận hồ sơ đăng ký cấp Giấy chứng nhận quyền sử dụng đất lần đầu tại Đồng Nai?',
    label='Q10 | Cấp sổ đỏ Đồng Nai — cơ quan'
)
results.append(r)

In [ ]:
r = ask(
    'Phí và lệ phí khi đăng ký cấp sổ đỏ lần đầu tại Đồng Nai là bao nhiêu?',
    label='Q11 | Cấp sổ đỏ Đồng Nai — phí lệ phí'
)
results.append(r)

## 5. DoD 3 — Thiếu jurisdiction → confirmation_needed

In [ ]:
r = ask(
    'Điều kiện chuyển mục đích sử dụng đất nông nghiệp sang đất ở là gì?',
    label='Q12 | Thiếu jurisdiction | DoD-3'
)
results.append(r)
assert r['confirmation_needed'] is True, 'Phải confirmation_needed=True khi thiếu jurisdiction'
assert r['confirmation_prompt'] is not None
print('✅ DoD 3 PASS')

## 6. DoD 4 — Negative test: khai sinh TP.HCM vs Đồng Nai

In [ ]:
r = ask(
    'Quy định đăng ký khai sinh tại TP.HCM khác Đồng Nai như thế nào?',
    label='Q13 | Negative — khai sinh địa phương | DoD-4'
)
results.append(r)
# Khai sinh là thủ tục toàn quốc → pipeline trả về empty context (chưa có data Hộ tịch)
# hoặc nói rõ không có sự khác biệt. Không được bịa sự khác biệt.
print('\n⚠️  Kiểm tra thủ công DoD 4: câu trả lời không được bịa sự khác biệt địa phương không tồn tại')

## 7. Gap 3 — câu hỏi cần traversal [:IMPLEMENTS]

In [ ]:
r = ask(
    'Nghị định 102/2024/NĐ-CP hướng dẫn thi hành Luật Đất đai 2024 quy định gì về chuyển mục đích sử dụng đất tại TP.HCM?',
    label='Q14 | Gap3 — NĐ 102 CMĐSDĐ TP.HCM'
)
results.append(r)

In [ ]:
r = ask(
    'Bảng giá đất TP.HCM năm 2025 ảnh hưởng như thế nào đến tiền sử dụng đất khi chuyển mục đích?',
    label='Q15 | Gap2+Gap3 — Bảng giá đất TP.HCM'
)
results.append(r)

## 8. Tổng kết kết quả

In [ ]:
print('\n' + '='*70)
print('TỔNG KẾT PIPELINE E2E TEST — TASK-14')
print('='*70)

total = len(results)
confirmed = sum(1 for r in results if r['confirmation_needed'])
answered = total - confirmed
with_parsed_citations = sum(1 for r in results if not r['confirmation_needed'] and r['citations'])
avg_elapsed = sum(r['elapsed_seconds'] for r in results) / total if total else 0

print(f'  Tổng câu hỏi:             {total}')
print(f'  Câu trả lời được:         {answered}')
print(f'  Cần xác nhận jurisdiction: {confirmed}')
print(f'  Có citation (format chuẩn): {with_parsed_citations}/{answered}')
print(f'  Thời gian TB:             {avg_elapsed:.1f}s')
if total:
    print(f'  Max elapsed:              {max(r["elapsed_seconds"] for r in results):.1f}s')

print('\nChi tiết:')
for i, r in enumerate(results, 1):
    if r['confirmation_needed']:
        status = '⚠️  CONFIRM'
    elif r['citations']:
        status = f'✅ {len(r["citations"])} cite'
    else:
        status = '⚡ 0 cite*'
    print(f'  Q{i:02d}: {status:12} {r["elapsed_seconds"]:5.1f}s | LCCIDs={r["lccids_count"]:4d} | {r["question"][:55]}')

print('\n* 0 cite = LLM có thể dùng format tắt, kiểm tra thủ công')

In [ ]:
# Đóng clients
neo4j_driver.close()
print('Clients closed.')